# Calendar Dimension Table

Builds the standard calendar table `caresync_databricks.silver.calendar` with one row per day from 1940-01-01 to 2030-12-31, including date key, year, quarter, month, week, day and flag attributes.

In [0]:
from pyspark.sql import functions as F

START_DATE = "1940-01-01"
END_DATE = "2030-12-31"
TARGET_TABLE = "caresync_databricks.silver.calendar"

# One row per day between START_DATE and END_DATE (inclusive)
dates_df = spark.sql(
    f"SELECT explode(sequence(DATE'{START_DATE}', DATE'{END_DATE}', INTERVAL 1 DAY)) AS date"
)

d = F.col("date")

calendar_df = (
    dates_df
    .select(
        F.date_format(d, "yyyyMMdd").cast("int").alias("datekey"),
        d.alias("date"),
        # Year
        F.year(d).alias("year"),
        F.when(F.month(d) <= 6, 1).otherwise(2).alias("half_year"),
        F.concat(F.lit("H"), F.when(F.month(d) <= 6, 1).otherwise(2)).alias("half_year_name"),
        # Quarter
        F.quarter(d).alias("quarter"),
        F.concat(F.lit("Q"), F.quarter(d)).alias("quarter_name"),
        F.concat(F.year(d), F.lit("-Q"), F.quarter(d)).alias("year_quarter"),
        # Month
        F.month(d).alias("month"),
        F.date_format(d, "MMMM").alias("month_name"),
        F.date_format(d, "MMM").alias("month_short_name"),
        F.date_format(d, "yyyy-MM").alias("year_month"),
        (F.year(d) * 100 + F.month(d)).alias("year_month_key"),
        # Week (ISO 8601)
        F.weekofyear(d).alias("week_of_year"),
        F.ceil(F.dayofmonth(d) / 7).cast("int").alias("week_of_month"),
        F.date_trunc("week", d).cast("date").alias("week_start_date"),
        F.date_add(F.date_trunc("week", d).cast("date"), 6).alias("week_end_date"),
        # Day
        F.dayofmonth(d).alias("day_of_month"),
        F.dayofyear(d).alias("day_of_year"),
        ((F.dayofweek(d) + 5) % 7 + 1).alias("day_of_week"),  # ISO: 1=Monday ... 7=Sunday
        F.date_format(d, "EEEE").alias("day_name"),
        F.date_format(d, "EEE").alias("day_short_name"),
        # Period boundaries
        F.trunc(d, "month").alias("month_start_date"),
        F.last_day(d).alias("month_end_date"),
        F.dayofmonth(F.last_day(d)).alias("days_in_month"),
        F.date_trunc("quarter", d).cast("date").alias("quarter_start_date"),
        F.date_sub(F.add_months(F.date_trunc("quarter", d).cast("date"), 3), 1).alias("quarter_end_date"),
        F.trunc(d, "year").alias("year_start_date"),
        F.make_date(F.year(d), F.lit(12), F.lit(31)).alias("year_end_date"),
        # Flags
        F.dayofweek(d).isin(1, 7).alias("is_weekend"),
        (~F.dayofweek(d).isin(1, 7)).alias("is_weekday"),
        (d == F.trunc(d, "month")).alias("is_month_start"),
        (d == F.last_day(d)).alias("is_month_end"),
        (d == F.date_trunc("quarter", d).cast("date")).alias("is_quarter_start"),
        (d == F.date_sub(F.add_months(F.date_trunc("quarter", d).cast("date"), 3), 1)).alias("is_quarter_end"),
        ((F.month(d) == 1) & (F.dayofmonth(d) == 1)).alias("is_year_start"),
        ((F.month(d) == 12) & (F.dayofmonth(d) == 31)).alias("is_year_end"),
        (((F.year(d) % 4 == 0) & (F.year(d) % 100 != 0)) | (F.year(d) % 400 == 0)).alias("is_leap_year"),
    )
)

display(calendar_df.limit(10))

In [0]:
(
    calendar_df
    .orderBy("date")
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

spark.sql(f"COMMENT ON TABLE {TARGET_TABLE} IS 'Standard calendar dimension, one row per day from {START_DATE} to {END_DATE}. datekey = yyyyMMdd.'")
print(f"Written {spark.table(TARGET_TABLE).count():,} rows to {TARGET_TABLE}")

In [0]:
cal = spark.table(TARGET_TABLE)

expected_days = spark.sql(f"SELECT datediff(DATE'{END_DATE}', DATE'{START_DATE}') + 1 AS n").first()["n"]

display(
    cal.agg(
        F.count("*").alias("row_count"),
        F.lit(expected_days).alias("expected_rows"),
        F.countDistinct("datekey").alias("distinct_datekeys"),
        F.min("date").alias("min_date"),
        F.max("date").alias("max_date"),
        F.sum((F.col("is_leap_year") & F.col("is_year_end")).cast("int")).alias("leap_years"),
        F.sum(F.col("is_weekend").cast("int")).alias("weekend_days"),
    )
)

# Spot-check known dates
display(cal.filter(F.col("date").isin("1940-01-01", "2024-02-29", "2026-11-08", "2030-12-31")))